In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from dotenv import load_dotenv
import os

In [52]:
load_dotenv("../.env")
census_key = os.getenv("CENSUS_API_KEY")

In [53]:
API_KEY    = census_key  
STATE_FIPS = "54"                         
YEARS      = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

VARIABLE   = "S2301_C02_001E,S2301_C04_001E"

records = []

In [54]:
for year in YEARS:
    url = (
        f"https://api.census.gov/data/{year}/acs/acs5/subject"
        f"?get=NAME,{VARIABLE}"
        f"&for=county:*"
        f"&in=state:{STATE_FIPS}"
        f"&key={API_KEY}"
    )
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    headers = data[0]
    for row in data[1:]:
        record = dict(zip(headers, row))
        record["Year"] = year
        records.append(record)

df = pd.DataFrame(records)

df


,NAME,S2301_C02_001E,S2301_C04_001E,state,county,Year
0,"Grant County, West Virginia",58.8,7.2,54,023,2013
1,"Summers County, West Virginia",42.4,7.9,54,089,2013
2,"Brooke County, West Virginia",56.0,10.1,54,009,2013
3,"Greenbrier County, West Virginia",53.6,7.5,54,025,2013
4,"Hardy County, West Virginia",56.5,10.6,54,031,2013
...,...,...,...,...,...,...
600,"Webster County, West Virginia",41.0,9.6,54,101,2023
601,"Wetzel County, West Virginia",46.6,5.8,54,103,2023
602,"Wirt County, West Virginia",48.1,3.3,54,105,2023
603,"Wood County, West Virginia",55.5,5.6,54,107,2023


In [55]:
# ── STEP 2: Clean up columns ─────────────────────────────────────────────────
df["FIPS_Code"]                      = df["state"] + df["county"]
df["County"]                         = df["NAME"].str.replace(", West Virginia", "", regex=False)
df["Labor_Force_Participation_Rate"] = pd.to_numeric(df["S2301_C02_001E"], errors="coerce")
df["Unemployment_Rate"]              = pd.to_numeric(df["S2301_C04_001E"], errors="coerce")

# ── STEP 3: Final structure ───────────────────────────────────────────────────
df = df[["Year", "FIPS_Code", "County", "Labor_Force_Participation_Rate", "Unemployment_Rate"]].copy()
df = df.sort_values(["Year", "FIPS_Code"]).reset_index(drop=True)

df

,Year,FIPS_Code,County,Labor_Force_Participation_Rate,Unemployment_Rate
0,2013,54001,Barbour County,51.8,8.0
1,2013,54003,Berkeley County,66.1,11.9
2,2013,54005,Boone County,46.7,10.3
3,2013,54007,Braxton County,50.1,13.5
4,2013,54009,Brooke County,56.0,10.1
...,...,...,...,...,...
600,2023,54101,Webster County,41.0,9.6
601,2023,54103,Wetzel County,46.6,5.8
602,2023,54105,Wirt County,48.1,3.3
603,2023,54107,Wood County,55.5,5.6


In [56]:
# ── RUCC: Load both vintages from xlsx ───────────────────────────────────────
def load_rucc_xlsx(path, rucc_col):
    """Read a RUCC xlsx, return a WV-only DataFrame with FIPS_Code and Rural_Urban_Continuum_Code."""
    raw = pd.read_excel(path, dtype={"FIPS": str, "FIPS_Code": str})

    # Normalise column names (strip whitespace)
    raw.columns = raw.columns.str.strip()

    # Pad FIPS to 5 digits
    fips_col = "FIPS" if "FIPS" in raw.columns else raw.columns[0]
    raw["FIPS_Code"] = raw[fips_col].str.zfill(5)

    # Filter to West Virginia (state FIPS 54)
    wv = raw[raw["FIPS_Code"].str.startswith("54")].copy()

    wv = wv[["FIPS_Code", rucc_col]].copy()
    wv.rename(columns={rucc_col: "Rural_Urban_Continuum_Code"}, inplace=True)
    wv["Rural_Urban_Continuum_Code"] = pd.to_numeric(wv["Rural_Urban_Continuum_Code"], errors="coerce")
    return wv

In [57]:
rucc_2013 = load_rucc_xlsx("../data/RUCC/ruralurbancodes2013.xlsx", "RUCC_2013")
rucc_2023 = load_rucc_xlsx("../data/RUCC/Ruralurbancontinuumcodes2023.xlsx", "RUCC_2023")

# ── RUCC: Year-conditional merge ─────────────────────────────────────────────
# 2013 vintage → years 2013-2022 | 2023 vintage → year 2023
df_pre2023  = df[df["Year"] <= 2022].merge(rucc_2013, on="FIPS_Code", how="left")
df_2023     = df[df["Year"] == 2023].merge(rucc_2023, on="FIPS_Code", how="left")

df = pd.concat([df_pre2023, df_2023], ignore_index=True)

In [58]:
# ── STEP 4: Final structure ───────────────────────────────────────────────────
# ── Final structure ───────────────────────────────────────────────────────────
df = df[["Year", "FIPS_Code", "County", "Labor_Force_Participation_Rate",
         "Unemployment_Rate", "Rural_Urban_Continuum_Code"]].copy()
df = df.sort_values(["Year", "FIPS_Code"]).reset_index(drop=True)

In [59]:
df

,Year,FIPS_Code,County,Labor_Force_Participation_Rate,Unemployment_Rate,Rural_Urban_Continuum_Code
0,2013,54001,Barbour County,51.8,8.0,6.0
1,2013,54003,Berkeley County,66.1,11.9,2.0
2,2013,54005,Boone County,46.7,10.3,3.0
3,2013,54007,Braxton County,50.1,13.5,8.0
4,2013,54009,Brooke County,56.0,10.1,3.0
...,...,...,...,...,...,...
600,2023,54101,Webster County,41.0,9.6,9.0
601,2023,54103,Wetzel County,46.6,5.8,7.0
602,2023,54105,Wirt County,48.1,3.3,3.0
603,2023,54107,Wood County,55.5,5.6,3.0


In [60]:
df.to_csv("unemployment_data.csv", index=False)